# 23 — Üçlü consensus ile model eğitimi

Bu notebook yalnızca eğitim akışını yürütür; iç/dış test değerlendirmesi bu aşamada yapılmaz.

Kullanılan veri: `Twitter Financial News Sentiment` + `Financial PhraseBank/allagree`.
Yalnızca `human_label == chatgpt_label == gemini_label` olan kayıtlar alınır.
Twitter consensus kayıtları train/validation/internal-test olarak ayrılır; PhraseBank/allagree consensus kayıtları yalnızca train'e eklenir.

Her model için checkpoint klasörü ayrıdır. Notebook yarıda kesilirse yeniden çalıştırıldığında son checkpoint'ten devam eder.

In [1]:
from thesis_utils import PROJECT_ROOT, DATA_ROOT, paths
from pathlib import Path
import gc, inspect, json, os, random
import numpy as np
import pandas as pd
import torch
from torch import nn
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from transformers.trainer_utils import get_last_checkpoint

os.environ['WANDB_DISABLED'] = 'true'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
LABEL2ID = {'negative': 0, 'neutral': 1, 'positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
RANDOM_STATE, MAX_LENGTH, EPOCHS = 42, 128, 4
TRAIN_BATCH_SIZE, EVAL_BATCH_SIZE = 16, 32
LEARNING_RATE, WEIGHT_DECAY, SAVE_TOTAL_LIMIT = 2e-5, 0.01, 3
USE_CLASS_WEIGHTS, FORCE_RETRAIN = True, False

TWITTER_PATH = DATA_ROOT / 'annotations' / 'reannotation_21' / 'final' / 'twitter_financial_news_reannotated.csv'
PHRASEBANK_PATH = DATA_ROOT / 'annotations' / 'reannotation_21' / 'final' / 'financial_phrasebank_reannotated_all_configs.csv'
SPLIT_ROOT = DATA_ROOT / 'splits' / 'consensus_twitter_phrasebank_allagree_v1'
CHECKPOINT_ROOT = paths.MODEL_CHECKPOINT_ROOT / 'consensus_twitter_phrasebank_allagree_v1'
RESULTS_ROOT = paths.MODEL_RESULTS_ROOT / 'consensus_twitter_phrasebank_allagree_v1'
for p in [SPLIT_ROOT, CHECKPOINT_ROOT, RESULTS_ROOT]: p.mkdir(parents=True, exist_ok=True)

MODEL_CONFIGS = [
    {'model_key': 'bert', 'display_name': 'BERT-base-uncased', 'model_name': 'bert-base-uncased', 'is_finbert': False},
    {'model_key': 'distilbert', 'display_name': 'DistilBERT-base-uncased', 'model_name': 'distilbert-base-uncased', 'is_finbert': False},
    {'model_key': 'roberta', 'display_name': 'RoBERTa-base', 'model_name': 'roberta-base', 'is_finbert': False},
    {'model_key': 'finbert', 'display_name': 'FinBERT', 'model_name': 'ProsusAI/finbert', 'is_finbert': True},
]

def set_all_seeds(seed=RANDOM_STATE):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_all_seeds()
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
print('Checkpoint root:', CHECKPOINT_ROOT)

Device: cpu
Checkpoint root: D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\checkpoints\financial_sentiment_multi_model\consensus_twitter_phrasebank_allagree_v1


In [2]:
def normalize_text(s):
    return s.astype(str).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

def consensus_mask(df):
    required = ['human_label', 'chatgpt_label', 'gemini_label']
    missing = [c for c in required if c not in df.columns]
    if missing: raise ValueError(f'Consensus kolonları eksik: {missing}')
    labels = df[required].astype(str).apply(lambda c: c.str.lower().str.strip())
    return labels.nunique(axis=1).eq(1)

def clean_source(df, dataset_name, config=None):
    out = df.copy()
    out['text'] = out['text'].astype(str).str.strip()
    for c in ['human_label', 'chatgpt_label', 'gemini_label']:
        out[c] = out[c].astype(str).str.lower().str.strip()
    out = out[(out['text'] != '') & (~out['text'].str.lower().isin(['nan', 'none']))]
    out = out[out['human_label'].isin(LABEL2ID)]
    out = out[consensus_mask(out)]
    if config is not None: out = out[out['config'].eq(config)]
    out['source_dataset'] = dataset_name
    out['text_key'] = normalize_text(out['text'])
    out = out.drop_duplicates('text_key').reset_index(drop=True)
    out['label'] = out['human_label']
    out['label_id'] = out['label'].map(LABEL2ID).astype(int)
    return out

if not TWITTER_PATH.exists() or not PHRASEBANK_PATH.exists():
    raise FileNotFoundError('Üçlü reannotation CSV dosyalarından biri bulunamadı.')
twitter = clean_source(pd.read_csv(TWITTER_PATH), 'TwitterFinancialNewsSentiment')
phrasebank = clean_source(pd.read_csv(PHRASEBANK_PATH), 'FinancialPhraseBank', config='allagree')

# Veri kümeleri arası aynı normalize metin tek kopyaya indirilir.
phrasebank = phrasebank[~phrasebank['text_key'].isin(set(twitter['text_key']))].copy()
twitter_train_val, internal_test = train_test_split(twitter, test_size=0.20, random_state=RANDOM_STATE, stratify=twitter['label_id'])
twitter_train, validation = train_test_split(twitter_train_val, test_size=0.15, random_state=RANDOM_STATE, stratify=twitter_train_val['label_id'])
train = pd.concat([twitter_train, phrasebank], ignore_index=True)

for a, b in [('train', train), ('validation', validation), ('internal_test', internal_test)]:
    b['input_text'] = b['text'].astype(str).str.strip()
    b['label'] = b['label'].astype(str).str.lower().str.strip()
    b['label_id'] = b['label'].map(LABEL2ID).astype(int)
    b.insert(0, 'sample_id', [f'{a}_{i:07d}' for i in range(len(b))])
    b.to_csv(SPLIT_ROOT / f'{a}.csv', index=False, encoding='utf-8-sig')

keys = {name: set(df['text_key']) for name, df in [('train', train), ('validation', validation), ('internal_test', internal_test)]}
if keys['train'] & keys['validation'] or keys['train'] & keys['internal_test'] or keys['validation'] & keys['internal_test']:
    raise ValueError('Splitler arasında text leakage bulundu.')
manifest = {'random_state': RANDOM_STATE, 'phrasebank_config': 'allagree', 'consensus_rule': 'human_label == chatgpt_label == gemini_label', 'models': MODEL_CONFIGS, 'counts': {k: len(v) for k, v in [('train', train), ('validation', validation), ('internal_test', internal_test)]}}
(SPLIT_ROOT / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
print('Twitter consensus:', len(twitter))
print('PhraseBank/allagree consensus:', len(phrasebank))
for name, df in [('train', train), ('validation', validation), ('internal_test', internal_test)]: print(name, df.shape, df['label'].value_counts().to_dict())

Twitter consensus: 8146
PhraseBank/allagree consensus: 2145
train (7683, 15) {'neutral': 4330, 'positive': 1986, 'negative': 1367}
validation (978, 14) {'neutral': 536, 'positive': 253, 'negative': 189}
internal_test (1630, 14) {'neutral': 893, 'positive': 423, 'negative': 314}


In [3]:
TEXT_COL, LABEL_ID_COL = 'input_text', 'label_id'

def prepare_hf_dataset(split_df, tokenizer):
    temp = split_df[[TEXT_COL, LABEL_ID_COL]].copy().rename(columns={LABEL_ID_COL: 'labels'})
    temp[TEXT_COL] = temp[TEXT_COL].astype(str)
    ds = Dataset.from_pandas(temp.reset_index(drop=True))
    ds = ds.map(lambda batch: tokenizer(batch[TEXT_COL], truncation=True, padding='max_length', max_length=MAX_LENGTH), batched=True)
    keep = ['input_ids', 'attention_mask', 'labels']
    if 'token_type_ids' in ds.column_names: keep.append('token_type_ids')
    ds.set_format(type='torch', columns=keep)
    return ds

def normalize_finbert_label(value):
    value = str(value).lower().strip()
    return {'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral', 'label_0': 'positive', 'label_1': 'negative', 'label_2': 'neutral'}.get(value, value)

def align_finbert_classifier(model):
    original = {int(k): normalize_finbert_label(v) for k, v in model.config.id2label.items()}
    if set(original.values()) != set(LABEL2ID): raise ValueError(f'Beklenmeyen FinBERT mapping: {original}')
    classifier, old_weight = model.classifier, model.classifier.weight.detach().clone()
    old_bias = classifier.bias.detach().clone() if classifier.bias is not None else None
    source_id = {label: idx for idx, label in original.items()}
    with torch.no_grad():
        for target_id, label in ID2LABEL.items():
            classifier.weight[target_id].copy_(old_weight[source_id[label]])
            if old_bias is not None: classifier.bias[target_id].copy_(old_bias[source_id[label]])
    model.config.id2label, model.config.label2id = ID2LABEL.copy(), LABEL2ID.copy()
    return model

def load_initial_model(cfg):
    if cfg['is_finbert']: return align_finbert_classifier(AutoModelForSequenceClassification.from_pretrained(cfg['model_name']))
    return AutoModelForSequenceClassification.from_pretrained(cfg['model_name'], num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True)

def trainer_tokenizer_kwargs(tokenizer):
    sig = inspect.signature(Trainer.__init__)
    if 'processing_class' in sig.parameters: return {'processing_class': tokenizer}
    if 'tokenizer' in sig.parameters: return {'tokenizer': tokenizer}
    return {}

def compute_metrics(eval_pred):
    logits, labels = eval_pred; preds = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, preds), 'f1_macro': f1_score(labels, preds, average='macro'), 'f1_weighted': f1_score(labels, preds, average='weighted')}

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights_tensor=None, **kwargs):
        super().__init__(*args, **kwargs); self.class_weights_tensor = class_weights_tensor
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.get('labels'); outputs = model(**inputs)
        weights = self.class_weights_tensor.to(outputs.logits.device) if self.class_weights_tensor is not None else None
        loss = nn.CrossEntropyLoss(weight=weights)(outputs.logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def build_training_args(output_dir):
    sig = inspect.signature(TrainingArguments.__init__)
    kwargs = dict(output_dir=str(output_dir), learning_rate=LEARNING_RATE, per_device_train_batch_size=TRAIN_BATCH_SIZE, per_device_eval_batch_size=EVAL_BATCH_SIZE, num_train_epochs=EPOCHS, weight_decay=WEIGHT_DECAY, logging_steps=50, report_to='none', seed=RANDOM_STATE)
    if 'data_seed' in sig.parameters: kwargs['data_seed'] = RANDOM_STATE
    if 'eval_strategy' in sig.parameters: kwargs['eval_strategy'] = 'epoch'
    elif 'evaluation_strategy' in sig.parameters: kwargs['evaluation_strategy'] = 'epoch'
    for key, value in [('save_strategy', 'epoch'), ('save_total_limit', SAVE_TOTAL_LIMIT), ('load_best_model_at_end', True), ('metric_for_best_model', 'f1_macro'), ('greater_is_better', True), ('logging_strategy', 'steps')]:
        if key in sig.parameters: kwargs[key] = value
    if 'fp16' in sig.parameters: kwargs['fp16'] = torch.cuda.is_available()
    return TrainingArguments(**kwargs)

counts = np.bincount(train['label_id'].astype(int), minlength=3)
CLASS_WEIGHTS = torch.tensor([len(train) / (3 * c) if c else 1.0 for c in counts], dtype=torch.float) if USE_CLASS_WEIGHTS else None
print('Class weights:', CLASS_WEIGHTS)

Class weights: tensor([1.8734, 0.5915, 1.2895])


In [4]:
def train_one_model(cfg):
    set_all_seeds()
    run_dir, checkpoint_dir = CHECKPOINT_ROOT / cfg['model_key'], CHECKPOINT_ROOT / cfg['model_key'] / 'checkpoints'
    final_dir, result_dir = run_dir / 'final_model', RESULTS_ROOT / cfg['model_key']
    marker = run_dir / 'training_complete.json'
    checkpoint_dir.mkdir(parents=True, exist_ok=True); result_dir.mkdir(parents=True, exist_ok=True)
    if marker.exists() and final_dir.exists() and not FORCE_RETRAIN:
        print(f'ATLANIYOR: {cfg["display_name"]} tamamlanmış.')
        return json.loads(marker.read_text(encoding='utf-8'))
    model_source = str(final_dir) if final_dir.exists() and not FORCE_RETRAIN else cfg['model_name']
    tokenizer = AutoTokenizer.from_pretrained(model_source)
    train_ds, val_ds = prepare_hf_dataset(train, tokenizer), prepare_hf_dataset(validation, tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(model_source) if model_source == str(final_dir) else load_initial_model(cfg)
    model.config.id2label, model.config.label2id = ID2LABEL.copy(), LABEL2ID.copy()
    trainer = WeightedTrainer(model=model, args=build_training_args(checkpoint_dir), train_dataset=train_ds, eval_dataset=val_ds, compute_metrics=compute_metrics, class_weights_tensor=CLASS_WEIGHTS, **trainer_tokenizer_kwargs(tokenizer))
    last_checkpoint = None if FORCE_RETRAIN else get_last_checkpoint(str(checkpoint_dir))
    if not final_dir.exists() or FORCE_RETRAIN:
        print(f'\nEĞİTİM: {cfg["display_name"]} | resume={last_checkpoint}')
        trainer.train(resume_from_checkpoint=last_checkpoint)
        final_dir.mkdir(parents=True, exist_ok=True); trainer.save_model(str(final_dir)); tokenizer.save_pretrained(str(final_dir))
    else: print(f'FINAL MODEL YÜKLENDİ: {cfg["display_name"]}')
    val_metrics = trainer.evaluate(eval_dataset=val_ds)
    summary = {'model_key': cfg['model_key'], 'model_name': cfg['model_name'], 'train_n': len(train), 'validation_n': len(validation), 'validation_loss': float(val_metrics.get('eval_loss', np.nan)), 'validation_f1_macro': float(val_metrics.get('eval_f1_macro', np.nan)), 'checkpoint_dir': str(checkpoint_dir), 'final_model_dir': str(final_dir)}
    (result_dir / 'training_summary.json').write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    marker.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8')
    del trainer, model, tokenizer, train_ds, val_ds; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return summary

summaries = []
for model_cfg in MODEL_CONFIGS:
    summaries.append(train_one_model(model_cfg))
    pd.DataFrame(summaries).to_csv(RESULTS_ROOT / 'training_summary_all_models.csv', index=False, encoding='utf-8-sig')
training_summary_df = pd.DataFrame(summaries)
display(training_summary_df)

Map:   0%|          | 0/7683 [00:00<?, ? examples/s]

Map:   0%|          | 0/978 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\Users\kayma\AppData


EĞİTİM: BERT-base-uncased | resume=None


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\kayma\AppData\Roaming\Python\Python314\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,F1 Macro,F1 Weighted
0.026088,0.339568,4,0.940695,0.934135,0.940706


,model_key,model_name,train_n,validation_n,validation_loss,validation_f1_macro,checkpoint_dir,final_model_dir
0,bert,bert-base-uncased,7683,978,0.339662,0.925104,D:\serkan.kaymak\financial_sentiment_thesis\fi...,D:\serkan.kaymak\financial_sentiment_thesis\fi...
1,distilbert,distilbert-base-uncased,7683,978,0.387481,0.911650,D:\serkan.kaymak\financial_sentiment_thesis\fi...,D:\serkan.kaymak\financial_sentiment_thesis\fi...
2,roberta,roberta-base,7683,978,0.235041,0.954411,D:\serkan.kaymak\financial_sentiment_thesis\fi...,D:\serkan.kaymak\financial_sentiment_thesis\fi...
3,finbert,ProsusAI/finbert,7683,978,0.339568,0.934135,D:\serkan.kaymak\financial_sentiment_thesis\fi...,D:\serkan.kaymak\financial_sentiment_thesis\fi...
